# Logan River SUMMA best-run evaluation

Evaluate and visualize the best calibrated `workshop_run_summa_asyncdds_1` SUMMA run without rerunning preprocessing or calibration. The notebook regenerates the optimizer final-evaluation output from the latest saved best parameters, aligns simulated discharge with USGS observations, and computes diagnostic metrics over the completed post-spinup period.

Native SUMMA runoff output is converted from `m s-1` to `m3/s` using `settings/SUMMA/attributes.nc` before comparison to observed streamflow.

In [ ]:
# Imports and paths
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import yaml

warnings.filterwarnings("ignore", message=".*is an EXPERIMENTAL module.*")

repo_dir = Path.cwd()
if repo_dir.name == ".ipynb_checkpoints":
    repo_dir = repo_dir.parent

config_path = repo_dir / "config_logan_river_lumped_summa_asyncdds.yaml"
if not config_path.exists():
    config_path = Path(
        "examples/04_workshop_notebooks/config_logan_river_lumped_summa_asyncdds.yaml"
    )

with config_path.open() as handle:
    config = yaml.safe_load(handle)

data_dir = Path(config["SYMFLUENCE_DATA_DIR"])
domain_name = config["DOMAIN_NAME"]
experiment_id = config["EXPERIMENT_ID"]
project_dir = data_dir / f"domain_{domain_name}"
settings_dir = project_dir / "settings" / "SUMMA"
opt_dir = project_dir / "optimization" / "SUMMA" / f"dds_{experiment_id}"
final_eval_dir = opt_dir / "final_evaluation"
obs_path = (
    project_dir
    / "data"
    / "observations"
    / "streamflow"
    / "preprocessed"
    / f"{domain_name}_streamflow_processed.csv"
)

print(f"Config: {config_path}")
print(f"Project: {project_dir}")
print(f"Best-run final evaluation: {final_eval_dir}")
print(f"Observations: {obs_path}")

In [ ]:
# Regenerate final evaluation from the best available optimizer record
import logging

from symfluence.models.summa.calibration.optimizer import SUMMAModelOptimizer

best_params_path = opt_dir / f"{experiment_id}_dds_best_params.json"
trace_path = opt_dir / f"{experiment_id}_parallel_iteration_results.csv"

best_params_data = {}
if best_params_path.exists():
    with best_params_path.open() as handle:
        best_params_data = json.load(handle)

best_params = best_params_data.get("best_params", {})
best_score = best_params_data.get("best_score", float("-inf"))
best_iteration = best_params_data.get("best_iteration")
best_parameter_source = best_params_path.name if best_params else None

# Prefer the iteration trace when it contains a better completed run
if trace_path.exists():
    trace = pd.read_csv(trace_path)
    if not trace.empty and "score" in trace.columns:
        trace_best = trace.loc[trace["score"].idxmax()]
        trace_best_score = float(trace_best["score"])
        if not best_params or trace_best_score >= float(best_score):
            param_keys = [
                param.strip()
                for config_key in ["PARAMS_TO_CALIBRATE", "BASIN_PARAMS_TO_CALIBRATE"]
                for param in str(config.get(config_key, "")).split(",")
                if param.strip()
            ]
            best_params = {
                param: float(trace_best[param])
                for param in param_keys
                if param in trace_best and pd.notna(trace_best[param])
            }
            best_score = trace_best_score
            best_iteration = int(trace_best["iteration"])
            best_parameter_source = trace_path.name

if not best_params:
    raise ValueError(
        f"No best parameters found in {best_params_path} or {trace_path}"
    )

logger = logging.getLogger("summa_best_run_evaluation")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

optimizer = SUMMAModelOptimizer(config, logger)
final_result = optimizer.run_final_evaluation(best_params)
if not final_result:
    raise RuntimeError("SUMMA final evaluation failed for the saved best parameters")

optimizer._save_final_evaluation_results(final_result, "DDS")

print(f"Best parameter source: {best_parameter_source}")
print(f"Best parameter score: {best_score}")
print(f"Best parameter iteration: {best_iteration}")
print(f"Regenerated final evaluation: {final_eval_dir}")

In [ ]:
# Load SUMMA discharge from the optimizer best-run output
def read_basin_area_m2(settings_dir: Path) -> float | None:
    attrs_path = settings_dir / "attributes.nc"
    if not attrs_path.exists():
        return None
    with xr.open_dataset(attrs_path) as attrs:
        if "HRUarea" not in attrs:
            return None
        return float(attrs["HRUarea"].sum())


def runoff_to_discharge(data_array: xr.DataArray, area_m2: float | None) -> xr.DataArray:
    runoff = data_array
    if "gru" in runoff.dims:
        runoff = runoff.isel(gru=0)
    if "hru" in runoff.dims:
        runoff = runoff.isel(hru=0)

    units = str(runoff.attrs.get("units", "")).replace(" ", "")
    if units in {"ms-1", "m/s"} and area_m2 is not None:
        runoff = runoff * area_m2
        runoff.attrs["units"] = "m3/s"
    return runoff


def read_summa_netcdf(path: Path, area_m2: float | None) -> pd.Series | None:
    with xr.open_dataset(path) as dataset:
        for variable in [
            "averageRoutedRunoff_mean",
            "averageRoutedRunoff",
            "scalarTotalRunoff",
        ]:
            if variable in dataset:
                discharge = runoff_to_discharge(dataset[variable], area_m2)
                series = discharge.to_series()
                series.index = pd.to_datetime(series.index).round("s")
                series.name = "simulated_cms"
                return series.sort_index()
    return None


basin_area_m2 = read_basin_area_m2(settings_dir)
best_record_paths = [best_params_path, trace_path]
best_record_mtime = max(
    path.stat().st_mtime for path in best_record_paths if path.exists()
)
candidate_netcdfs = []
candidate_netcdfs.extend(sorted(final_eval_dir.glob("*timestep*.nc")))
candidate_netcdfs.extend(sorted(final_eval_dir.glob("*.nc")))

sim = None
sim_source = None
for netcdf_path in dict.fromkeys(candidate_netcdfs):
    if "day" in netcdf_path.stem:
        continue
    if netcdf_path.stat().st_mtime < best_record_mtime:
        continue
    sim = read_summa_netcdf(netcdf_path, basin_area_m2)
    if sim is not None:
        sim_source = netcdf_path
        break

if sim is None:
    raise FileNotFoundError(
        "No current best-run SUMMA final-evaluation output found. "
        "Run the regeneration cell above before plotting."
    )

print(f"Source: {sim_source}")
print(f"Basin area: {basin_area_m2:.1f} m2" if basin_area_m2 else "Basin area: missing")
print(f"Rows read: {len(sim)}")
print(f"Time range: {sim.index.min()} to {sim.index.max()}")
print(sim.describe().to_string())

In [ ]:
# Read and align USGS observations
obs = pd.read_csv(obs_path)
obs["datetime"] = pd.to_datetime(obs["datetime"], utc=True, errors="coerce")
obs["datetime"] = obs["datetime"].dt.tz_convert(None)
obs = obs.dropna(subset=["datetime"]).set_index("datetime").sort_index()

sim_daily = sim.resample("D").mean()
obs_daily = obs["discharge_cms"].resample("D").mean()
common_index = sim_daily.index.intersection(obs_daily.index)

aligned = pd.DataFrame(
    {
        "observed_cms": obs_daily.loc[common_index],
        "simulated_cms": sim_daily.loc[common_index],
    }
).dropna()

spinup_end = pd.to_datetime(config["SPINUP_PERIOD"].split(",")[1].strip())
eval_aligned = aligned.loc[aligned.index > spinup_end]

print(f"Aligned days including spinup: {len(aligned)}")
print(f"Aligned days after spinup: {len(eval_aligned)}")
print(f"Evaluation window available: {eval_aligned.index.min()} to {eval_aligned.index.max()}")

In [ ]:
# Hydrologic metrics over the completed post-spinup period
def nse(obs_values: pd.Series, sim_values: pd.Series) -> float:
    denominator = ((obs_values - obs_values.mean()) ** 2).sum()
    if denominator == 0:
        return np.nan
    return 1.0 - ((sim_values - obs_values) ** 2).sum() / denominator


def kge(obs_values: pd.Series, sim_values: pd.Series) -> tuple[float, float, float, float]:
    if len(obs_values) < 2:
        return np.nan, np.nan, np.nan, np.nan
    r = np.corrcoef(obs_values, sim_values)[0, 1]
    alpha = sim_values.std(ddof=1) / obs_values.std(ddof=1)
    beta = sim_values.mean() / obs_values.mean()
    value = 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)
    return value, r, alpha, beta


def percent_bias(obs_values: pd.Series, sim_values: pd.Series) -> float:
    return 100.0 * (sim_values.sum() - obs_values.sum()) / obs_values.sum()


obs_eval = eval_aligned["observed_cms"]
sim_eval = eval_aligned["simulated_cms"]
kge_value, corr, variability_ratio, bias_ratio = kge(obs_eval, sim_eval)
metrics = {
    "NSE": nse(obs_eval, sim_eval),
    "KGE": kge_value,
    "Correlation": corr,
    "Variability ratio": variability_ratio,
    "Bias ratio": bias_ratio,
    "PBIAS (%)": percent_bias(obs_eval, sim_eval),
    "RMSE (cms)": np.sqrt(((sim_eval - obs_eval) ** 2).mean()),
    "Mean observed (cms)": obs_eval.mean(),
    "Mean simulated (cms)": sim_eval.mean(),
}

pd.Series(metrics).to_frame("value")

In [ ]:
# Streamflow diagnostic plots
plot_data = eval_aligned.copy()
if plot_data.empty:
    raise ValueError("No post-spinup overlap is available")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    f"Logan River SUMMA best-run evaluation\n{sim_source}",
    fontsize=14,
    fontweight="bold",
)

# Time series
axes[0, 0].plot(plot_data.index, plot_data["observed_cms"], label="Observed (USGS)", linewidth=1.0)
axes[0, 0].plot(plot_data.index, plot_data["simulated_cms"], label="SUMMA", linewidth=1.0)
axes[0, 0].set_title("Daily streamflow")
axes[0, 0].set_ylabel("Discharge (m3/s)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].text(
    0.02,
    0.95,
    f"NSE: {metrics['NSE']:.3f}\n"
    f"KGE: {metrics['KGE']:.3f}\n"
    f"Bias: {metrics['PBIAS (%)']:.1f}%",
    transform=axes[0, 0].transAxes,
    va="top",
    bbox={"facecolor": "white", "alpha": 0.85},
)

# Scatter
axes[0, 1].scatter(plot_data["observed_cms"], plot_data["simulated_cms"], s=12, alpha=0.45)
max_flow = max(plot_data["observed_cms"].max(), plot_data["simulated_cms"].max())
axes[0, 1].plot([0, max_flow], [0, max_flow], "k--", alpha=0.5)
axes[0, 1].set_title("Observed vs simulated")
axes[0, 1].set_xlabel("Observed (m3/s)")
axes[0, 1].set_ylabel("Simulated (m3/s)")
axes[0, 1].grid(alpha=0.3)

# Monthly means for available period
monthly = plot_data.groupby(plot_data.index.month).mean(numeric_only=True)
axes[1, 0].plot(monthly.index, monthly["observed_cms"], "o-", label="Observed")
axes[1, 0].plot(monthly.index, monthly["simulated_cms"], "o-", label="SUMMA")
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_title("Monthly mean flow for completed period")
axes[1, 0].set_ylabel("Discharge (m3/s)")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Flow duration curve
obs_sorted = np.sort(plot_data["observed_cms"].to_numpy())[::-1]
sim_sorted = np.sort(plot_data["simulated_cms"].to_numpy())[::-1]
obs_exceedance = np.arange(1, len(obs_sorted) + 1) / (len(obs_sorted) + 1) * 100.0
sim_exceedance = np.arange(1, len(sim_sorted) + 1) / (len(sim_sorted) + 1) * 100.0
axes[1, 1].semilogy(obs_exceedance, obs_sorted, label="Observed")
axes[1, 1].semilogy(sim_exceedance, sim_sorted, label="SUMMA")
axes[1, 1].set_title("Flow duration curve")
axes[1, 1].set_xlabel("Exceedance probability (%)")
axes[1, 1].set_ylabel("Discharge (m3/s)")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calibration progress and saved final-evaluation metrics
trace_path = opt_dir / f"{experiment_id}_parallel_iteration_results.csv"
final_metrics_path = opt_dir / f"{experiment_id}_dds_final_evaluation.json"

if trace_path.exists():
    trace = pd.read_csv(trace_path)
    trace["best_score"] = trace["score"].cummax()
    display(trace.tail())

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(trace["iteration"], trace["best_score"], linewidth=2)
    ax.scatter(trace["iteration"].iloc[-1], trace["best_score"].iloc[-1], color="red", zorder=5)
    ax.set_title("DDS calibration progress")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(f"Best {config.get('OPTIMIZATION_METRIC', 'score')}")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"No calibration trace found: {trace_path}")

if final_metrics_path.exists():
    final_metrics = json.loads(final_metrics_path.read_text())
    display(pd.DataFrame(final_metrics.get("calibration_metrics", {}), index=["calibration"]).T)
    display(pd.DataFrame(final_metrics.get("evaluation_metrics", {}), index=["evaluation"]).T)
else:
    print(f"No final evaluation JSON found: {final_metrics_path}")